# 09 store

Question: does `golem-store` keep the schema that `okf/store.md` lists, answer every call of the
plugins and refuse every call that breaks a rule of the store, above all the accounts with an
amount that is negative or zero, an overdraw and an unknown account, and does its file hold every
change after a close, a prefix of the changes after a kill between changes and after a writer that
fails, refuse every file that is no store of this Golem, and answer the calls of the server thread
far inside the budget of 1 ms.

`golem-devtool store` runs a JSON script of calls on a store and prints the answer of every step
as a line as soon as it ran, `store-describe` prints the schema of a file as SQLite reports it,
and `store-bench` times the calls of the server thread on a filled store. The notebook holds every
answer to the expectation written beside its step, so a section reads as a list of calls and what
each must answer. The players are made up UUIDs and every file lies in a temporary directory.

In [ ]:
# Papermill overrides this cell.
# The tree under test, the root of the repository.
golem_root = "."

# The wiki page whose schema the file must match, and the seed of the permissions, both repository
# relative.
wiki = "okf/store.md"
seed = "config/permissions.toml"

# The rounds of changes before the flush and in the burst that the kill cuts, and the limit of the
# file size, in KiB, at which the writer must fail.
flushed_rounds = 1000
burst_rounds = 20000
size_limit_kib = 300

# The size of the store of the cost section, the calls of each kind it times, and the budget of a
# call of the server thread in microseconds, DEFAULT_BUDGET of golem-plugin.
bench_players = 5000
bench_repeat = 2000
budget_us = 1000

## Setup

The cell finds the repository, builds `golem-devtool` and prints the versions of Python, cargo,
rustc and the SQLite that `rusqlite` bundles. `steps` runs a script of pairs of a step and its
expectation and prints one row per step: an expectation is the answer itself, `ANY` for any
answer, a `Fails` with the kind and the message of the error, or a function that judges the
answer.

In [ ]:
import hashlib
import json
import os
import re
import resource
import signal
import sqlite3
import subprocess
import sys
import tempfile
import tomllib
from pathlib import Path

for _base in [Path.cwd(), *Path.cwd().parents]:
    if (_base / "tests" / "harness" / "__init__.py").is_file():
        sys.path.insert(0, str(_base / "tests"))
        break

from harness import QUIET, paths, run

P = paths(golem_root)
SCRATCH = Path(tempfile.mkdtemp(prefix="golem-09-"))
ENVIRONMENT = dict(os.environ, **QUIET)

run(["cargo", "build", "--quiet", "-p", "golem-devtool"], cwd=P.root, timeout=1800, check=True)
metadata = json.loads(run(["cargo", "metadata", "--format-version", "1"], cwd=P.root, check=True).stdout)
DEVTOOL = Path(metadata["target_directory"]) / "debug" / "golem-devtool"
assert DEVTOOL.is_file(), "cargo built no golem-devtool"
PACKAGES = {package["name"]: package["version"] for package in metadata["packages"]}

ALEX = "a1e5a1e5-0000-4000-8000-000000000001"
SAM = "5a305a30-0000-4000-8000-000000000002"
KIM = "c1a0c1a0-0000-4000-8000-000000000003"
ESSIE = "e55e0000-0000-4000-8000-000000000004"
NOBODY = "00000000-0000-4000-8000-0000000000ff"
NAMES = {ALEX: "Alex", SAM: "Sam", KIM: "Kim", ESSIE: "Essie", NOBODY: "nobody"}
LARGEST = 2**63 - 1
COUNTS = {"steps": 0, "refusals": 0, "scripts": 0}


class Any:
    def __repr__(self):
        return "ANY"


ANY = Any()


class Fails:
    """An expected error: its kind and its message, or a pattern the message must match."""

    def __init__(self, kind, message=None, pattern=None):
        self.kind, self.message, self.pattern = kind, message, pattern

    def __repr__(self):
        return "Fails(%r, %r)" % (self.kind, self.message or self.pattern)


def refused(message):
    return Fails("refused", message)


def place(dimension="minecraft:overworld", x=0.5, y=64.0, z=0.5, yaw=0.0, pitch=0.0):
    return {"dimension": dimension, "x": x, "y": y, "z": z, "yaw": yaw, "pitch": pitch}


def shown(text):
    text = text if isinstance(text, str) else json.dumps(text)
    for uuid, name in NAMES.items():
        text = text.replace(uuid, name)
    return text.replace(str(SCRATCH), "<scratch>")


def script(steps, stdin=None, preexec=None):
    COUNTS["scripts"] += 1
    path = SCRATCH / ("script-%03d.json" % COUNTS["scripts"])
    path.write_text(json.dumps({"steps": steps}), encoding="utf-8")
    result = subprocess.run([str(DEVTOOL), "store", path.name], cwd=SCRATCH, env=ENVIRONMENT, input=stdin,
                            capture_output=True, text=True, timeout=1200, preexec_fn=preexec)
    assert result.returncode == 0, "store exited %d\n%s" % (result.returncode, result.stderr)
    answers = [json.loads(line) for line in result.stdout.splitlines()]
    assert [answer["step"] for answer in answers] == list(range(len(steps))), result.stdout[-2000:]
    return answers, result.stderr


def judge(expected, answer):
    if isinstance(expected, Fails):
        error = answer.get("error")
        if not error or error["kind"] != expected.kind:
            return False
        if expected.kind == "refused":
            COUNTS["refusals"] += 1
        if expected.pattern is not None:
            return re.fullmatch(expected.pattern, error["message"]) is not None
        return expected.message is None or error["message"] == expected.message
    if "ok" not in answer:
        return False
    if expected is ANY:
        return True
    if callable(expected):
        return bool(expected(answer["ok"]))
    return answer["ok"] == expected


def steps(pairs, preexec=None):
    answers, stderr = script([step for step, _ in pairs], preexec=preexec)
    wrong = []
    for (step, expected), answer in zip(pairs, answers):
        good = judge(expected, answer)
        COUNTS["steps"] += 1
        outcome = answer["ok"] if "ok" in answer else answer["error"]
        print("%-3s %-26s %s" % ("ok" if good else "BAD", step["op"], shown(outcome)[:150]))
        if not good:
            wrong.append("step %d %s: expected %s, got %s" % (answer["step"], step["op"], expected, shown(answer)))
    assert not wrong, "\n".join(wrong)
    return [answer.get("ok") for answer in answers], stderr


def devtool(*arguments):
    return subprocess.run([str(DEVTOOL), *map(str, arguments)], cwd=SCRATCH, env=ENVIRONMENT,
                          capture_output=True, text=True, timeout=1200)


def open_store(path="store.db"):
    return ({"op": "open", "path": path}, ANY)


CLOSE = ({"op": "close"}, ANY)

print("python:", sys.version.split()[0])
print("cargo:", run(["cargo", "--version"], cwd=P.root, check=True).stdout.strip())
print("rustc:", run(["rustc", "--version"], cwd=P.root, check=True).stdout.strip())
print("rusqlite:", PACKAGES.get("rusqlite"), "libsqlite3-sys:", PACKAGES.get("libsqlite3-sys"))

## The schema

A new file becomes a store of version 1 with the application id `Golm`, in the journal mode WAL,
and every table of the store is STRICT and has the columns, the keys and the foreign keys that the
section The schema of `okf/store.md` lists in its tables, no more and no fewer.

In [ ]:
answers, _ = steps([open_store(), CLOSE])
described = devtool("store-describe", "store.db")
assert described.returncode == 0, described.stderr
SCHEMA = json.loads(described.stdout)


def wiki_tables():
    text = (P.root / wiki).read_text(encoding="utf-8")
    section = text.split("\n## The schema\n", 1)[1].split("\n## ", 1)[0]
    tables = {}
    for block in section.split("\n### ")[1:]:
        name, _, body = block.partition("\n")
        rows = [line for line in body.splitlines() if line.startswith("| `")]
        columns, keys = [], []
        for row in rows:
            column, kind, null, key, references, _ = [cell.strip() for cell in row.strip().strip("|").split("|")]
            column = column.strip("`")
            columns.append({"name": column, "type": kind, "not_null": null == "no",
                            "primary_key": int(key) if key else 0})
            if references:
                table, _, to = references.strip("`").partition(".")
                keys.append({"from": column, "table": table, "to": to})
        tables[name.strip()] = {"columns": columns, "foreign_keys": keys}
    return tables


WIKI = wiki_tables()
assert SCHEMA["application_id"] == 0x476F6C6D, SCHEMA["application_id"]
assert SCHEMA["version"] == 1, SCHEMA["version"]
assert SCHEMA["journal_mode"] == "wal", SCHEMA["journal_mode"]
assert SCHEMA["indexes"] == ["mail_by_receiver", "transfers_by_source", "transfers_by_target"], SCHEMA["indexes"]
names = [table["name"] for table in SCHEMA["tables"]]
assert sorted(WIKI) == names, "the file has the tables %s, the wiki %s" % (names, sorted(WIKI))
for table in SCHEMA["tables"]:
    page = WIKI[table["name"]]
    assert table["strict"], "%s is not STRICT" % table["name"]
    assert table["rows"] == 0, table
    assert table["columns"] == page["columns"], "%s: the file has %s, the wiki %s" % (
        table["name"], table["columns"], page["columns"])
    order = lambda keys: sorted((key["from"], key["table"], key["to"]) for key in keys)
    assert order(table["foreign_keys"]) == order(page["foreign_keys"]), "%s: %s against %s" % (
        table["name"], table["foreign_keys"], page["foreign_keys"])
    key = sorted((column["primary_key"], column["name"]) for column in table["columns"] if column["primary_key"])
    references = ", ".join("%s to %s.%s" % reference for reference in order(table["foreign_keys"])) or "none"
    print("%-26s %2d columns, key %s, foreign keys %s" % (
        table["name"], len(table["columns"]), ", ".join(name for _, name in key), references))

## The players

A first join creates a player, a later one sets the name, the last join and the last seen time
and names the name before. The calls of a quit, of `/back`, of AFK and of a nickname change their
field alone, a lookup by name takes the name of the last join in any case, and `put_player` puts
the whole record of the migration tool. A UUID is read in either case and written in lower case.
A call for a player who never joined, a bad name and a bad location are refused and change
nothing.

In [ ]:
LOGOUT = place(x=100.25, y=70.0, z=-30.75, yaw=180.0, pitch=15.5)
BACK = place("minecraft:the_nether", x=-8.0, y=32.0, z=12.0, yaw=-90.0, pitch=0.0)
ALEX_RECORD = {"uuid": ALEX, "name": "Alexis", "first_join": 1000, "last_join": 2000, "last_seen": 3000,
               "logout": LOGOUT, "back": BACK, "afk": True, "nickname": "Lexi"}
SAM_RECORD = {"uuid": SAM, "name": "Sam", "first_join": 1100, "last_join": 1100, "last_seen": 1100,
              "logout": None, "back": None, "afk": False, "nickname": "lexi"}
ESSIE_RECORD = {"uuid": ESSIE, "name": "Essie", "first_join": 500, "last_join": 600, "last_seen": 700,
                "logout": place("minecraft:the_end", x=1.0, y=2.0, z=3.0), "back": None, "afk": False,
                "nickname": None}
unknown = refused("the player %s is not in the store" % NOBODY)
answers, _ = steps([
    open_store(),
    ({"op": "join", "uuid": ALEX, "name": "Alex", "at": 1000}, {"first": True, "previous_name": None}),
    ({"op": "join", "uuid": SAM, "name": "Sam", "at": 1100}, {"first": True, "previous_name": None}),
    ({"op": "join", "uuid": KIM, "name": "Kim", "at": 1200}, {"first": True, "previous_name": None}),
    ({"op": "join", "uuid": ALEX.upper(), "name": "Alexis", "at": 2000}, {"first": False, "previous_name": "Alex"}),
    ({"op": "quit", "uuid": ALEX, "location": LOGOUT, "at": 3000}, None),
    ({"op": "set_back", "uuid": ALEX, "location": BACK}, None),
    ({"op": "set_afk", "uuid": ALEX, "afk": True}, None),
    ({"op": "set_nickname", "uuid": ALEX, "nickname": "Lexi"}, None),
    ({"op": "set_nickname", "uuid": SAM, "nickname": "lexi"}, None),
    ({"op": "set_back", "uuid": SAM, "location": BACK}, None),
    ({"op": "set_back", "uuid": SAM, "location": None}, None),
    ({"op": "set_last_seen", "uuid": KIM, "at": 4000}, None),
    ({"op": "put_player", **ESSIE_RECORD}, None),
    ({"op": "player", "uuid": ALEX}, ALEX_RECORD),
    ({"op": "player", "uuid": SAM}, SAM_RECORD),
    ({"op": "player", "uuid": ESSIE}, ESSIE_RECORD),
    ({"op": "player", "uuid": KIM}, lambda record: record["last_seen"] == 4000 and record["last_join"] == 1200),
    ({"op": "player", "uuid": NOBODY}, None),
    ({"op": "player_named", "name": "ALEXIS"}, ALEX_RECORD),
    ({"op": "player_named", "name": "Alex"}, None),
    ({"op": "players_nicknamed", "nickname": "LEXI"}, [ALEX_RECORD, SAM_RECORD]),
    ({"op": "player_count"}, 4),
    ({"op": "quit", "uuid": NOBODY, "location": LOGOUT, "at": 3000}, unknown),
    ({"op": "set_afk", "uuid": NOBODY, "afk": True}, unknown),
    ({"op": "join", "uuid": ALEX, "name": "", "at": 5000}, refused("the name of a player is empty")),
    ({"op": "join", "uuid": ALEX, "name": "x" * 65, "at": 5000},
     refused("the name of a player has 65 characters, more than 64")),
    ({"op": "join", "uuid": ALEX, "name": "Al\nex", "at": 5000},
     refused("the name of a player holds the control character '\\n'")),
    ({"op": "set_nickname", "uuid": ALEX, "nickname": "y" * 65},
     refused("a nickname has 65 characters, more than 64")),
    ({"op": "quit", "uuid": ALEX, "location": place("minecraft overworld"), "at": 5000},
     refused("the logout location names the dimension \"minecraft overworld\", which holds the character ' '")),
    ({"op": "quit", "uuid": ALEX, "location": place(""), "at": 5000},
     refused("the logout location names the dimension \"\", which is empty")),
    ({"op": "quit", "uuid": ALEX, "location": place(x="NaN"), "at": 5000},
     refused("the logout location has the x NaN, which is not a finite number")),
    ({"op": "set_back", "uuid": ALEX, "location": place(z="inf")},
     refused("the back location has the z inf, which is not a finite number")),
    ({"op": "player", "uuid": "steve"}, Fails("argument", '"uuid" "steve" is not a UUID')),
    ({"op": "player", "uuid": ALEX}, ALEX_RECORD),
    CLOSE,
])

## Homes, warps and spawns

A home belongs to a player and is found by its name byte for byte, so `Base` and `base` are two
homes. Setting a home, a warp or a spawn answers whether it replaced one, and removing answers
whether there was one. A warp carries the node a player needs for it, if any. The spawn of the
group `default` is the one spawn of the server.

In [ ]:
BASE = place(x=10.5, y=64.0, z=-3.5, yaw=90.0)
BASE2 = place(x=11.5, y=65.0, z=-4.5, yaw=45.0, pitch=-10.0)
UPPER = place(x=0.0, y=80.0, z=0.0)
NETHER = place("minecraft:the_nether", x=1.0, y=40.0, z=1.0)
SHOP = place(x=200.0, y=70.0, z=200.0)
VAULT = place(x=-500.0, y=12.0, z=-500.0, yaw=270.0)
SPAWN = place(x=0.5, y=100.0, z=0.5)
ADMIN_SPAWN = place(x=50.5, y=100.0, z=50.5)
HOMES = {"base": BASE2, "nether": NETHER}
WARPS = [{"name": "vault", "location": VAULT, "permission": "essentials.warps.vault"}]
SPAWNS = {"default": SPAWN}
answers, _ = steps([
    open_store(),
    ({"op": "set_home", "uuid": ALEX, "name": "base", "location": BASE}, False),
    ({"op": "set_home", "uuid": ALEX, "name": "base", "location": BASE2}, True),
    ({"op": "set_home", "uuid": ALEX, "name": "Base", "location": UPPER}, False),
    ({"op": "set_home", "uuid": ALEX, "name": "nether", "location": NETHER}, False),
    ({"op": "home", "uuid": ALEX, "name": "base"}, BASE2),
    ({"op": "homes", "uuid": ALEX}, {"Base": UPPER, **HOMES}),
    ({"op": "home_count", "uuid": ALEX}, 3),
    ({"op": "remove_home", "uuid": ALEX, "name": "Base"}, True),
    ({"op": "remove_home", "uuid": ALEX, "name": "Base"}, False),
    ({"op": "homes", "uuid": ALEX}, HOMES),
    ({"op": "home", "uuid": SAM, "name": "base"}, None),
    ({"op": "homes", "uuid": NOBODY}, {}),
    ({"op": "set_home", "uuid": NOBODY, "name": "base", "location": BASE}, unknown),
    ({"op": "remove_home", "uuid": NOBODY, "name": "base"}, unknown),
    ({"op": "set_home", "uuid": ALEX, "name": "", "location": BASE}, refused("the name of a home is empty")),
    ({"op": "set_home", "uuid": ALEX, "name": "h" * 65, "location": BASE},
     refused("the name of a home has 65 characters, more than 64")),
    ({"op": "set_home", "uuid": ALEX, "name": "base", "location": place(x="NaN")},
     refused("the location of a home has the x NaN, which is not a finite number")),
    ({"op": "set_warp", "name": "shop", "location": BASE, "permission": None}, False),
    ({"op": "set_warp", "name": "vault", "location": VAULT, "permission": "essentials.warps.vault"}, False),
    ({"op": "set_warp", "name": "shop", "location": SHOP, "permission": None}, True),
    ({"op": "warp", "name": "vault"}, WARPS[0]),
    ({"op": "warps"}, [{"name": "shop", "location": SHOP, "permission": None}, *WARPS]),
    ({"op": "set_warp", "name": "bad", "location": BASE, "permission": "essentials warps"},
     refused("the permission of a warp \"essentials warps\" holds the character ' '")),
    ({"op": "set_warp", "name": "", "location": BASE}, refused("the name of a warp is empty")),
    ({"op": "remove_warp", "name": "shop"}, True),
    ({"op": "remove_warp", "name": "shop"}, False),
    ({"op": "warp", "name": "shop"}, None),
    ({"op": "set_spawn", "group": "default", "location": BASE}, False),
    ({"op": "set_spawn", "group": "admin", "location": ADMIN_SPAWN}, False),
    ({"op": "set_spawn", "group": "default", "location": SPAWN}, True),
    ({"op": "spawn", "group": "default"}, SPAWN),
    ({"op": "spawn", "group": "builder"}, None),
    ({"op": "spawns"}, {"admin": ADMIN_SPAWN, "default": SPAWN}),
    ({"op": "set_spawn", "group": "", "location": SPAWN}, refused("the group of a spawn is empty")),
    ({"op": "remove_spawn", "group": "admin"}, True),
    ({"op": "remove_spawn", "group": "admin"}, False),
    CLOSE,
])

## Mail and ignore lists

A mail comes from a player or from the console, which has no UUID, and gets the next number of
the store. Marking the mail of a player read answers how many were unread, and a mail can be
removed alone or with all others of its receiver. An ignore list answers whether a call changed it.
A mail to or from a player who never joined, an empty or too long text, a player who ignores
themselves and an unknown player on an ignore list are refused.

In [ ]:
def mail(number, sender, sender_name, receiver, text, sent_at, read):
    return {"id": number, "sender": sender, "sender_name": sender_name, "receiver": receiver, "text": text,
            "sent_at": sent_at, "read": read}


FIRST = mail(1, SAM, "Sam", ALEX, "hello Alex", 5000, False)
CONSOLE = mail(2, None, "Console", ALEX, "the server restarts at noon", 5100, False)
TO_SAM = mail(3, KIM, "Kim", SAM, "hi Sam", 5200, False)
answers, _ = steps([
    open_store(),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": ALEX, "text": "hello Alex", "at": 5000}, 1),
    ({"op": "send_mail", "sender": None, "sender_name": "Console", "receiver": ALEX,
      "text": "the server restarts at noon", "at": 5100}, 2),
    ({"op": "send_mail", "sender": KIM, "sender_name": "Kim", "receiver": SAM, "text": "hi Sam", "at": 5200}, 3),
    ({"op": "unread_mail", "receiver": ALEX}, 2),
    ({"op": "mail", "receiver": ALEX}, [FIRST, CONSOLE]),
    ({"op": "mark_mail_read", "receiver": ALEX}, 2),
    ({"op": "mark_mail_read", "receiver": ALEX}, 0),
    ({"op": "unread_mail", "receiver": ALEX}, 0),
    ({"op": "mail", "receiver": ALEX}, [dict(FIRST, read=True), dict(CONSOLE, read=True)]),
    ({"op": "remove_mail", "receiver": ALEX, "id": 1}, True),
    ({"op": "remove_mail", "receiver": ALEX, "id": 1}, False),
    ({"op": "remove_mail", "receiver": ALEX, "id": 3}, False),
    ({"op": "mail", "receiver": ALEX}, [dict(CONSOLE, read=True)]),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": ALEX, "text": "another", "at": 5300}, 4),
    ({"op": "clear_mail", "receiver": ALEX}, 2),
    ({"op": "clear_mail", "receiver": ALEX}, 0),
    ({"op": "mail", "receiver": ALEX}, []),
    ({"op": "mail", "receiver": SAM}, [TO_SAM]),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": NOBODY, "text": "hi", "at": 5400}, unknown),
    ({"op": "send_mail", "sender": NOBODY, "sender_name": "Ghost", "receiver": ALEX, "text": "boo", "at": 5400},
     unknown),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": ALEX, "text": "", "at": 5400},
     refused("the text of a mail is empty")),
    ({"op": "send_mail", "sender": SAM, "sender_name": "Sam", "receiver": ALEX, "text": "x" * 65537, "at": 5400},
     refused("the text of a mail has 65537 bytes, more than 65536")),
    ({"op": "send_mail", "sender": SAM, "sender_name": "", "receiver": ALEX, "text": "hi", "at": 5400},
     refused("the name of the sender is empty")),
    ({"op": "mark_mail_read", "receiver": NOBODY}, unknown),
    ({"op": "ignore", "player": ALEX, "other": SAM}, True),
    ({"op": "ignore", "player": ALEX, "other": SAM}, False),
    ({"op": "ignore", "player": ALEX, "other": KIM}, True),
    ({"op": "ignores", "player": ALEX, "other": SAM}, True),
    ({"op": "ignores", "player": SAM, "other": ALEX}, False),
    ({"op": "ignore_list", "player": ALEX}, [SAM, KIM]),
    ({"op": "unignore", "player": ALEX, "other": KIM}, True),
    ({"op": "unignore", "player": ALEX, "other": KIM}, False),
    ({"op": "ignore", "player": ALEX, "other": ALEX}, refused("the player %s cannot ignore themselves" % ALEX)),
    ({"op": "ignore", "player": ALEX, "other": NOBODY}, unknown),
    ({"op": "ignore", "player": NOBODY, "other": ALEX}, unknown),
    CLOSE,
])

## The accounts

An account opens with a starting balance, which the log records as a deposit when it is above
zero. A deposit adds money the server creates, a withdrawal takes money the server destroys, a
transfer moves money between two accounts in one step, and `set_balance` moves the difference, so
every change of a balance has its entry in the log. Every answer names the entry and the balances
after it, and `check` holds every balance to the sum of its log.

In [ ]:
def entry(number, at, source, target, amount, note):
    return {"id": number, "at": at, "source": source, "target": target, "amount": amount, "note": note}


def receipt(transfer, source=None, target=None):
    return {"transfer": transfer, "source_balance": source, "target_balance": target}


LEDGER_ALEX = [entry(7, 6500, ALEX, None, 500, "eco set"), entry(3, 6100, ALEX, SAM, 2500, "pay"),
               entry(1, 6000, None, ALEX, 10000, "starting balance")]
BALANCES = {ALEX: 7000, SAM: 2600, KIM: 1000}
answers, _ = steps([
    open_store(),
    ({"op": "create_account", "uuid": ALEX, "balance": 10000, "at": 6000, "note": "starting balance"},
     receipt(1, target=10000)),
    ({"op": "create_account", "uuid": SAM, "balance": 0, "at": 6000, "note": "starting balance"},
     receipt(None, target=0)),
    ({"op": "create_account", "uuid": KIM, "balance": 500, "at": 6000, "note": "starting balance"},
     receipt(2, target=500)),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 2500, "at": 6100, "note": "pay"},
     receipt(3, 7500, 2500)),
    ({"op": "deposit", "uuid": SAM, "amount": 100, "at": 6200, "note": "eco give"}, receipt(4, target=2600)),
    ({"op": "withdraw", "uuid": KIM, "amount": 200, "at": 6300, "note": "eco take"}, receipt(5, source=300)),
    ({"op": "set_balance", "uuid": KIM, "balance": 1000, "at": 6400, "note": "eco set"}, receipt(6, target=1000)),
    ({"op": "set_balance", "uuid": ALEX, "balance": 7000, "at": 6500, "note": "eco set"}, receipt(7, source=7000)),
    ({"op": "set_balance", "uuid": SAM, "balance": 2600, "at": 6600, "note": "eco set"}, receipt(None, target=2600)),
    ({"op": "balance", "uuid": ALEX}, 7000),
    ({"op": "balance", "uuid": SAM}, 2600),
    ({"op": "balance", "uuid": KIM}, 1000),
    ({"op": "balance", "uuid": ESSIE}, None),
    ({"op": "top_balances", "limit": 2}, [[ALEX, 7000], [SAM, 2600]]),
    ({"op": "top_balances", "limit": 10}, [[ALEX, 7000], [SAM, 2600], [KIM, 1000]]),
    ({"op": "account_count"}, 3),
    ({"op": "ledger", "account": ALEX, "limit": 10}, LEDGER_ALEX),
    ({"op": "ledger", "account": None, "limit": 3}, lambda entries: [e["id"] for e in entries] == [7, 6, 5]),
    ({"op": "check"}, lambda check: check["passed"] and check["rows"]["transfers"] == 7),
    CLOSE,
])

## The refusals of the accounts

Every change of an account that would break a rule is refused before anything moves: an amount
that is negative or zero, a balance below zero, an overdraw, an account that does not exist, a
transfer from an account to itself, a balance above the largest integer of 64 bits, a second
account of a player and a note that is too long. The checks run in the order the section
Refusals of `okf/store.md` names: the note, then the amount or the balance to set, then the same
account twice, then the accounts, the source first, then whether the source holds the amount and
the target can take it. So a change with a bad note is refused for its note whatever its amount,
and a transfer of zero cents to an unknown account is refused for its amount. After every refusal
the balances and the log are as they were.

In [ ]:
no_account = lambda uuid: refused("the player %s has no account" % uuid)
not_above = lambda amount: refused("the amount of %d cents is not above zero" % amount)
answers, _ = steps([
    open_store(),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": -5, "at": 7000, "note": "pay"}, not_above(-5)),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 0, "at": 7000, "note": "pay"}, not_above(0)),
    ({"op": "deposit", "uuid": ALEX, "amount": 0, "at": 7000}, not_above(0)),
    ({"op": "deposit", "uuid": ALEX, "amount": -1, "at": 7000}, not_above(-1)),
    ({"op": "withdraw", "uuid": ALEX, "amount": 0, "at": 7000}, not_above(0)),
    ({"op": "withdraw", "uuid": ALEX, "amount": -LARGEST - 1, "at": 7000}, not_above(-LARGEST - 1)),
    ({"op": "create_account", "uuid": ESSIE, "balance": -1, "at": 7000},
     refused("the balance of -1 cents is below zero")),
    ({"op": "set_balance", "uuid": ALEX, "balance": -1, "at": 7000},
     refused("the balance of -1 cents is below zero")),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 7001, "at": 7000, "note": "pay"},
     refused("the account of %s holds 7000 cents, less than the 7001 cents to take" % ALEX)),
    ({"op": "transfer", "source": SAM, "target": ALEX, "amount": LARGEST, "at": 7000, "note": "pay"},
     refused("the account of %s holds 2600 cents, less than the %d cents to take" % (SAM, LARGEST))),
    ({"op": "withdraw", "uuid": KIM, "amount": 1001, "at": 7000},
     refused("the account of %s holds 1000 cents, less than the 1001 cents to take" % KIM)),
    ({"op": "transfer", "source": ALEX, "target": ESSIE, "amount": 1, "at": 7000}, no_account(ESSIE)),
    ({"op": "transfer", "source": ESSIE, "target": ALEX, "amount": 1, "at": 7000}, no_account(ESSIE)),
    ({"op": "transfer", "source": ALEX, "target": NOBODY, "amount": 1, "at": 7000}, no_account(NOBODY)),
    ({"op": "transfer", "source": ALEX, "target": NOBODY, "amount": 0, "at": 7000}, not_above(0)),
    ({"op": "deposit", "uuid": NOBODY, "amount": 1, "at": 7000}, no_account(NOBODY)),
    ({"op": "withdraw", "uuid": NOBODY, "amount": 1, "at": 7000}, no_account(NOBODY)),
    ({"op": "set_balance", "uuid": NOBODY, "balance": 5, "at": 7000}, no_account(NOBODY)),
    ({"op": "transfer", "source": ALEX, "target": ALEX, "amount": 1, "at": 7000},
     refused("a transfer from the account of %s to itself moves nothing" % ALEX)),
    ({"op": "transfer", "source": NOBODY, "target": NOBODY, "amount": 5, "at": 7000},
     refused("a transfer from the account of %s to itself moves nothing" % NOBODY)),
    ({"op": "deposit", "uuid": SAM, "amount": LARGEST - 2599, "at": 7000},
     refused("the account of %s holds 2600 cents, and %d more would pass the largest balance of %d cents"
             % (SAM, LARGEST - 2599, LARGEST))),
    ({"op": "create_account", "uuid": ALEX, "balance": 0, "at": 7000},
     refused("the player %s has an account already" % ALEX)),
    ({"op": "create_account", "uuid": NOBODY, "balance": 0, "at": 7000}, unknown),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 1, "at": 7000, "note": "x" * 65537},
     refused("the note of a transfer has 65537 bytes, more than 65536")),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 1.5, "at": 7000},
     Fails("argument", '"amount" is not an integer')),
    ({"op": "create_account", "uuid": ESSIE, "balance": -1, "at": 7000, "note": "x" * 65537},
     refused("the note of a transfer has 65537 bytes, more than 65536")),
    ({"op": "set_balance", "uuid": ALEX, "balance": -1, "at": 7000, "note": "a\u0000b"},
     refused("the note of a transfer holds a NUL character")),
    ({"op": "transfer", "source": ALEX, "target": SAM, "amount": 0, "at": 7000, "note": "a\u0000b"},
     refused("the note of a transfer holds a NUL character")),
    ({"op": "deposit", "uuid": NOBODY, "amount": -1, "at": 7000, "note": "a\u0000b"},
     refused("the note of a transfer holds a NUL character")),
    ({"op": "balance", "uuid": ALEX}, BALANCES[ALEX]),
    ({"op": "balance", "uuid": SAM}, BALANCES[SAM]),
    ({"op": "balance", "uuid": KIM}, BALANCES[KIM]),
    ({"op": "ledger", "account": None, "limit": 100}, lambda entries: len(entries) == 7),
    ({"op": "check"}, lambda check: check["passed"]),
    CLOSE,
])

## The permissions

The seed `config/permissions.toml` goes into the store as a whole set, and the store answers it as
the same TOML, which `golem-devtool permissions` loads into the resolver of `golem-permissions`, so
a check of the set from the store decides every node of the seed as a check of the seed itself
does, with the same reason. A group and a player change alone and move the generation. The store
refuses what would break its rows: a group a list names but the store lacks, a node or a group
twice in a list, a name with white space, and the removal of a group that a group or a player
still names, and a player key of the TOML that is no UUID in lower case, as `golem-permissions`
refuses it. The cycles and the syntax of a node are the business of `golem-permissions`. A group,
the group of a spawn and a dimension are tokens of up to 256 bytes, so a group may have a longer
name than the 64 characters of a home.

In [ ]:
SEED_TEXT = (P.root / seed).read_text(encoding="utf-8")
SEED = tomllib.loads(SEED_TEXT)
(SCRATCH / "seed.toml").write_text(SEED_TEXT, encoding="utf-8")
SAVED = {}


def keep(name, test=lambda value: True):
    def judge_and_keep(value):
        SAVED[name] = value
        return test(value)
    return judge_and_keep


def seeded(value):
    groups = {group["name"]: group for group in value["groups"]}
    return (value["generation"] == 1 and sorted(groups) == ["admin", "default"] and value["players"] == []
            and groups["default"]["nodes"] == SEED["groups"]["default"]["nodes"]
            and groups["admin"] == {"name": "admin", "weight": 100, "parents": ["default"],
                                    "nodes": ["essentials.*", "golem.*"]})


NOT_A_UUID = "the key is not a UUID in lower case with its four dashes, such as 00000000-0000-4000-8000-000000000001"
ALEX_PERMISSIONS = {"uuid": ALEX, "groups": ["admin"], "nodes": ["-essentials.suicide"]}
BUILDER = {"name": "builder", "weight": 20, "parents": ["default"], "nodes": ["essentials.setwarp", "-essentials.suicide"]}
answers, _ = steps([
    open_store(),
    ({"op": "permissions"}, lambda value: value["groups"] == [] and value["players"] == [] and value["generation"] == 0),
    ({"op": "replace_permissions", "toml_file": "seed.toml"}, None),
    ({"op": "permissions"}, keep("seed", seeded)),
    ({"op": "put_player_permissions", **ALEX_PERMISSIONS}, None),
    ({"op": "player_permissions", "uuid": ALEX}, ALEX_PERMISSIONS),
    ({"op": "put_permission_group", "name": "builder", "weight": 10, "parents": ["default"],
      "nodes": ["essentials.setwarp"]}, False),
    ({"op": "put_permission_group", **BUILDER}, True),
    ({"op": "permission_group", "name": "builder"}, BUILDER),
    ({"op": "put_player_permissions", "uuid": SAM, "groups": ["builder"], "nodes": []}, None),
    ({"op": "permissions"}, lambda value: value["generation"] == 5 and len(value["players"]) == 2),
    ({"op": "put_permission_group", "name": "mod", "parents": ["moderator"]},
     refused('the group "moderator" is not in the permissions')),
    ({"op": "put_player_permissions", "uuid": KIM, "groups": ["vip"]}, refused('the group "vip" is not in the permissions')),
    ({"op": "put_permission_group", "name": "x", "nodes": ["essentials.home", "essentials.home"]},
     refused('the nodes of the group x name "essentials.home" twice')),
    ({"op": "put_player_permissions", "uuid": KIM, "groups": ["admin", "admin"]},
     refused('the groups of the player %s name "admin" twice' % KIM)),
    ({"op": "put_permission_group", "name": "bad group"},
     refused("the name of a group \"bad group\" holds the character ' '")),
    ({"op": "put_permission_group", "name": "y", "nodes": ["essentials home"]},
     refused("a node of a group \"essentials home\" holds the character ' '")),
    ({"op": "put_permission_group", "name": "builder", "weight": "10"}, Fails("argument", '"weight" is not an integer')),
    ({"op": "put_permission_group", "name": "builder", "weight": 1.5}, Fails("argument", '"weight" is not an integer')),
    ({"op": "put_permission_group", "name": "builder", "weight": 2**31},
     Fails("argument", '"weight" does not fit 32 bits')),
    ({"op": "permission_group", "name": "builder"}, BUILDER),
    ({"op": "remove_permission_group", "name": "default"},
     refused('the group "default" cannot be removed, since the group admin names it')),
    ({"op": "remove_permission_group", "name": "builder"},
     refused('the group "builder" cannot be removed, since the player %s names it' % SAM)),
    ({"op": "replace_permissions", "toml": "[groups.default]\n[players.steve]\ngroups = []\n"},
     Fails("argument", "players.steve: " + NOT_A_UUID)),
    ({"op": "replace_permissions", "toml": "[groups.default]\n[players.%s]\nnodes = [\"x.y\"]\n" % KIM.upper()},
     Fails("argument", "players.%s: %s" % (KIM.upper(), NOT_A_UUID))),
    ({"op": "replace_permissions", "toml": "[groups.default]\n[players.\"a b\"]\n"},
     Fails("argument", 'players."a b": ' + NOT_A_UUID)),
    ({"op": "replace_permissions", "toml": "[groups.default]\n[groups.admin]\ninherits = [\"moderator\"]\n"},
     refused('the group "moderator" is not in the permissions')),
    ({"op": "put_player_permissions", "uuid": SAM, "groups": [], "nodes": []}, None),
    ({"op": "player_permissions", "uuid": SAM}, None),
    ({"op": "remove_permission_group", "name": "builder"}, True),
    ({"op": "remove_permission_group", "name": "builder"}, False),
    ({"op": "remove_player_permissions", "uuid": SAM}, False),
    ({"op": "permissions"}, keep("final", lambda value: value["generation"] == 7)),
    CLOSE,
])
answers, _ = steps([
    open_store("tokens.db"),
    ({"op": "join", "uuid": ALEX, "name": "Alex", "at": 1000}, ANY),
    ({"op": "put_permission_group", "name": "g" * 65}, False),
    ({"op": "put_permission_group", "name": "g" * 256}, False),
    ({"op": "put_permission_group", "name": "g" * 257},
     refused('the name of a group "%s" has 257 bytes, more than 256' % ("g" * 257))),
    ({"op": "set_spawn", "group": "s" * 256, "location": place()}, False),
    ({"op": "set_spawn", "group": "s" * 257, "location": place()},
     refused('the group of a spawn "%s" has 257 bytes, more than 256' % ("s" * 257))),
    ({"op": "set_home", "uuid": ALEX, "name": "deep", "location": place("d" * 256)}, False),
    ({"op": "set_home", "uuid": ALEX, "name": "deep", "location": place("d" * 257)},
     refused('the location of a home names the dimension "%s", which has 257 bytes, more than 256' % ("d" * 257))),
    CLOSE,
])

The set from the store and the seed must decide alike. The cell writes the TOML that the store
answered into a file and asks `golem-devtool permissions` for every node of the group default of
the seed and for admin nodes, for a player of the group default alone, once against the seed and
once against the TOML of the store. Alex is in the group admin in the store and denies
`essentials.suicide` of their own, which the resolver must read from the TOML of the store.

In [ ]:
STORE_TOML = SCRATCH / "store-permissions.toml"
STORE_TOML.write_text(SAVED["final"]["toml"], encoding="utf-8")
(SCRATCH / "store-seed.toml").write_text(SAVED["seed"]["toml"], encoding="utf-8")
NODES = SEED["groups"]["default"]["nodes"]
probe = [node.replace("*", "buy") for node in NODES] + ["essentials.setspawn", "essentials.eco", "golem.perm"]


def decisions(config, player):
    arguments = ["permissions", "--config", config, "--player", player]
    for node in probe:
        arguments += ["--node", node]
    result = devtool(*arguments)
    assert result.returncode == 0, "permissions refused %s: %s" % (config, result.stderr)
    return json.loads(result.stdout)["decisions"]


from_seed = decisions("seed.toml", KIM)
for name in ("store-seed.toml", "store-permissions.toml"):
    from_store = decisions(name, KIM)
    assert from_store == from_seed, "the set of %s decides otherwise:\n%s\n%s" % (name, from_store, from_seed)
assert sum(decision["value"] is True for decision in from_seed) == len(NODES), from_seed
alex = {decision["node"]: decision for decision in decisions("store-permissions.toml", ALEX)}
assert alex["essentials.setspawn"] == {"node": "essentials.setspawn", "value": True, "reason": "Group admin grants essentials.*"}, alex
assert alex["essentials.suicide"] == {"node": "essentials.suicide", "value": False, "reason": "The player denies essentials.suicide"}, alex
assert alex["golem.perm"]["value"] is True, alex
print("%d nodes decided alike from the seed and from the store, and Alex holds the admin nodes but not essentials.suicide"
      % len(probe))

## The settings, and a closed store

A setting is a text by key. A key that is empty and a value that is too long or holds a NUL are
refused. After `close` every change, `flush` and `check` answer that the store is closed, a read
still answers from the cache, and a second close does nothing.

In [ ]:
SETTINGS = {"motd": "Welcome back"}
answers, _ = steps([
    open_store(),
    ({"op": "set_setting", "key": "motd", "value": "Welcome to SkyTekx"}, None),
    ({"op": "set_setting", "key": "rules", "value": "Be kind"}, None),
    ({"op": "set_setting", "key": "motd", "value": "Welcome back"}, None),
    ({"op": "setting", "key": "motd"}, "Welcome back"),
    ({"op": "setting", "key": "none"}, None),
    ({"op": "settings"}, {"motd": "Welcome back", "rules": "Be kind"}),
    ({"op": "remove_setting", "key": "rules"}, True),
    ({"op": "remove_setting", "key": "rules"}, False),
    ({"op": "set_setting", "key": "", "value": "x"}, refused("the key of a setting is empty")),
    ({"op": "set_setting", "key": "k", "value": "x" * 65537},
     refused("the value of a setting has 65537 bytes, more than 65536")),
    ({"op": "set_setting", "key": "k", "value": "a\u0000b"}, refused("the value of a setting holds a NUL character")),
    ({"op": "close"}, lambda stats: stats["closed"] and stats["pending"] == 0 and stats["failure"] is None
     and stats["written"] == stats["queued"]),
    ({"op": "set_setting", "key": "motd", "value": "late"}, Fails("closed", "the store is closed")),
    ({"op": "flush"}, Fails("closed", "the store is closed")),
    ({"op": "check"}, Fails("closed", "the store is closed")),
    ({"op": "setting", "key": "motd"}, "Welcome back"),
    ({"op": "close"}, lambda stats: stats["closed"]),
])

## A reopen after a close

Every script above opened the same file and closed it, so each one already read what the one
before wrote. This cell reads everything once more, closes the store, opens it again and reads it
all a second time. Both readings must equal what the sections above left, `check` must find the
file sound, its accounts equal to their log and its rows equal to the cache, and the next mail
must take the number after the largest the store ever gave, though that mail was removed.

In [ ]:
READS = [
    ({"op": "player", "uuid": ALEX}, ALEX_RECORD),
    ({"op": "player", "uuid": ESSIE}, ESSIE_RECORD),
    ({"op": "player_count"}, 4),
    ({"op": "homes", "uuid": ALEX}, HOMES),
    ({"op": "warps"}, WARPS),
    ({"op": "spawns"}, SPAWNS),
    ({"op": "mail", "receiver": ALEX}, []),
    ({"op": "mail", "receiver": SAM}, [TO_SAM]),
    ({"op": "ignore_list", "player": ALEX}, [SAM]),
    ({"op": "balance", "uuid": ALEX}, BALANCES[ALEX]),
    ({"op": "balance", "uuid": SAM}, BALANCES[SAM]),
    ({"op": "top_balances", "limit": 10}, [[ALEX, 7000], [SAM, 2600], [KIM, 1000]]),
    ({"op": "ledger", "account": ALEX, "limit": 10}, LEDGER_ALEX),
    ({"op": "permissions"}, lambda value: value["toml"] == SAVED["final"]["toml"]),
    ({"op": "settings"}, SETTINGS),
    ({"op": "check"}, lambda check: check["passed"] and check["cache_differences"] == []),
]
answers, _ = steps([open_store(), *READS, ({"op": "reopen"}, ANY), *READS,
                    ({"op": "stats"}, lambda stats: stats["queued"] == 0),
                    ({"op": "send_mail", "sender": KIM, "sender_name": "Kim", "receiver": ALEX, "text": "after",
                      "at": 8000}, 5),
                    CLOSE])
before, after = answers[1:1 + len(READS)], answers[2 + len(READS):2 + 2 * len(READS)]
assert before[:-1] == after[:-1], "the reopen changed what the store answers"

## A reopen after a kill between changes

A process opens a new store, makes two players with accounts, runs 1000 rounds of a setting and a
transfer of one cent, flushes, and then queues a burst of 20000 more rounds without a flush. The
notebook kills it with `SIGKILL` the moment the burst is queued, while the writer is still
committing it. The next open must find a sound file that holds every round before the flush and a
prefix of the burst: the setting of a round is in the file only if the transfer of every round
before it is, and the transfer of a round only with its setting. The money of the two accounts
must still sum to the start.

In [ ]:
A, B = ALEX, SAM
ROUND = lambda key, note: [{"op": "set_setting", "key": key, "value": "{i}"},
                           {"op": "transfer", "source": A, "target": B, "amount": 1, "at": 9000, "note": note}]
KILLED = [
    {"op": "open", "path": "kill.db"},
    {"op": "join", "uuid": A, "name": "Alex", "at": 9000},
    {"op": "join", "uuid": B, "name": "Sam", "at": 9000},
    {"op": "create_account", "uuid": A, "balance": 1000000, "at": 9000, "note": "start"},
    {"op": "create_account", "uuid": B, "balance": 0, "at": 9000},
    {"op": "repeat", "count": flushed_rounds, "steps": ROUND("flushed", "flushed {i}")},
    {"op": "flush"},
    {"op": "repeat", "count": burst_rounds, "steps": ROUND("burst", "burst {i}")},
    {"op": "wait"},
]
(SCRATCH / "kill.json").write_text(json.dumps({"steps": KILLED}), encoding="utf-8")
process = subprocess.Popen([str(DEVTOOL), "store", "kill.json"], cwd=SCRATCH, env=ENVIRONMENT, stdin=subprocess.PIPE,
                           stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
seen = []
for line in process.stdout:
    seen.append(json.loads(line))
    if seen[-1]["step"] == 7:
        break
os.kill(process.pid, signal.SIGKILL)
process.wait(timeout=60)
assert process.returncode == -signal.SIGKILL, process.returncode
assert all("ok" in answer for answer in seen), seen
print("the devtool queued the %d calls of the burst, %.1f microseconds a call with the reading of its steps, "
      "and got SIGKILL" % (seen[-1]["ok"]["calls"], seen[-1]["ok"]["nanos_per_call"] / 1000))
answers, _ = steps([
    open_store("kill.db"),
    ({"op": "check"}, lambda check: check["passed"]),
    ({"op": "setting", "key": "flushed"}, str(flushed_rounds)),
    ({"op": "setting", "key": "burst"}, ANY),
    ({"op": "balance", "uuid": A}, ANY),
    ({"op": "balance", "uuid": B}, ANY),
    ({"op": "ledger", "account": B, "limit": 1}, ANY),
    CLOSE,
])
_, _, _, burst, balance_a, balance_b, last, _ = answers
rounds = balance_b - flushed_rounds
settings_in = int(burst) if burst is not None else 0
assert balance_a + balance_b == 1000000, "the accounts hold %d cents, not 1000000" % (balance_a + balance_b)
assert 0 <= rounds < burst_rounds, "the file holds %d rounds of the burst, so the kill came after the writer" % rounds
assert rounds in (settings_in, settings_in - 1), "the file holds %d transfers but the setting of round %d" % (rounds, settings_in)
note = "burst %d" % rounds if rounds else "flushed %d" % flushed_rounds
assert last[0]["note"] == note and last[0]["id"] == 1 + flushed_rounds + rounds, last
print("the file holds the %d rounds before the flush and %d of the %d rounds of the burst, the setting of round %d,"
      " and %d cents in both accounts" % (flushed_rounds, rounds, burst_rounds, settings_in, balance_a + balance_b))

## A second process

While a process holds a store, a second process can neither read nor open it, so the cache of the
first stays the truth of the file. The first process opens a store and waits on its standard
input. `store-describe` and the open of a second script must answer that another process holds the
file. Once the standard input of the first ends, it closes the store, and the file opens again.

In [ ]:
HOLDER = SCRATCH / "hold.json"
HOLDER.write_text(json.dumps({"steps": [{"op": "open", "path": "kill.db"}, {"op": "wait"}]}), encoding="utf-8")
holder = subprocess.Popen([str(DEVTOOL), "store", HOLDER.name], cwd=SCRATCH, env=ENVIRONMENT, stdin=subprocess.PIPE,
                          stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
assert json.loads(holder.stdout.readline())["op"] == "open"
assert json.loads(holder.stdout.readline())["op"] == "wait"
held = "cannot open the store kill.db: another process holds the file, so it is open in a second Golem or a tool"
described = devtool("store-describe", "kill.db")
assert described.returncode == 1 and described.stderr == "golem-devtool: store-describe: %s\n" % held, described
answers, _ = steps([({"op": "open", "path": "kill.db"}, Fails("open", held))])
holder.stdin.close()
assert holder.wait(timeout=60) == 0, holder.stderr.read()
assert devtool("store-describe", "kill.db").returncode == 0

## A reader that goes away

`store` and `store-describe` write their answers on the standard output. When the reader of that
pipe is gone, each must stop with the exit code 1 and one line on the standard error instead of a
panic, and `store` must still close its store, so the steps that ran reach the file. The cell
gives `store-describe` a pipe whose reader is closed, and closes the reader of a script while the
script waits, so its next answer finds no reader.

In [ ]:
read_end, write_end = os.pipe()
os.close(read_end)
gone = subprocess.run([str(DEVTOOL), "store-describe", "kill.db"], cwd=SCRATCH, env=ENVIRONMENT, stdout=write_end,
                      stderr=subprocess.PIPE, text=True, timeout=600)
os.close(write_end)
print("store-describe:", gone.returncode, gone.stderr.strip())
assert gone.returncode == 1 and "panicked" not in gone.stderr, gone.stderr
assert re.fullmatch(r"golem-devtool: store-describe: the standard output is closed: .+\n", gone.stderr), gone.stderr
(SCRATCH / "gone.json").write_text(json.dumps({"steps": [
    {"op": "open", "path": "gone.db"},
    {"op": "set_setting", "key": "before", "value": "the wait"},
    {"op": "wait"},
    {"op": "set_setting", "key": "after", "value": "the wait"},
    {"op": "set_setting", "key": "never", "value": "run"},
]}), encoding="utf-8")
reader = subprocess.Popen([str(DEVTOOL), "store", "gone.json"], cwd=SCRATCH, env=ENVIRONMENT, stdin=subprocess.PIPE,
                          stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
assert [json.loads(reader.stdout.readline())["op"] for _ in range(3)] == ["open", "set_setting", "wait"]
reader.stdout.close()
reader.stdin.close()
stderr = reader.stderr.read()
print("store:", reader.wait(timeout=600), stderr.strip())
assert reader.returncode == 1 and "panicked" not in stderr, stderr
assert re.fullmatch(r"golem-devtool: store: the standard output is closed: .+\n", stderr), stderr
answers, _ = steps([
    open_store("gone.db"),
    ({"op": "setting", "key": "before"}, "the wait"),
    ({"op": "setting", "key": "after"}, "the wait"),
    ({"op": "setting", "key": "never"}, None),
    CLOSE,
])

## A writer that fails

The process runs with a limit of 300 KiB on the size of the files it writes, as a full disk would
stop it, and with `SIGXFSZ` ignored, so a write past the limit fails instead of ending the process.
It flushes two accounts and then runs rounds of a setting and a transfer with a long note until
the writer fails. The calls after the failure, the flush and the close must answer that the store
takes no more changes, the log must hold the line of the failure, and the next open without the
limit must find a sound file that holds a prefix of the rounds, as after a kill.

In [ ]:
def limited():
    signal.signal(signal.SIGXFSZ, signal.SIG_IGN)
    resource.setrlimit(resource.RLIMIT_FSIZE, (size_limit_kib * 1024, size_limit_kib * 1024))


failure = r"the store takes no more changes, since the writer failed: .+ on the change \w+"
answers, stderr = steps([
    open_store("full.db"),
    ({"op": "join", "uuid": A, "name": "Alex", "at": 9000}, ANY),
    ({"op": "join", "uuid": B, "name": "Sam", "at": 9000}, ANY),
    ({"op": "create_account", "uuid": A, "balance": 1000000, "at": 9000, "note": "start"}, ANY),
    ({"op": "create_account", "uuid": B, "balance": 0, "at": 9000}, ANY),
    ({"op": "flush"}, None),
    ({"op": "repeat", "count": 100000, "steps": ROUND("counter", "round {i} " + "x" * 200)},
     Fails("failed", pattern=r"round \d+: " + failure)),
    ({"op": "flush"}, Fails("failed", pattern=failure)),
    ({"op": "stats"}, lambda stats: stats["failure"] and stats["lost"] > 0 and stats["pending"] == 0
     and stats["written"] + stats["lost"] == stats["queued"]),
    ({"op": "set_setting", "key": "after", "value": "the failure"}, Fails("failed", pattern=failure)),
    ({"op": "balance", "uuid": B}, ANY),
    ({"op": "close"}, Fails("failed", pattern=failure)),
    ({"op": "set_setting", "key": "after", "value": "the close"}, Fails("closed", "the store is closed")),
], preexec=limited)
assert re.search(r"^\[golem\] store: the writer failed and the store takes no more changes: .+$", stderr, re.M), stderr
assert (SCRATCH / "full.db").stat().st_size <= size_limit_kib * 1024
cached = answers[10]
answers, _ = steps([
    open_store("full.db"),
    ({"op": "check"}, lambda check: check["passed"]),
    ({"op": "setting", "key": "counter"}, ANY),
    ({"op": "balance", "uuid": B}, ANY),
    ({"op": "setting", "key": "after"}, None),
    CLOSE,
])
counter, rounds = int(answers[2] or 0), answers[3]
assert rounds in (counter, counter - 1), "the file holds %d transfers but the setting of round %d" % (rounds, counter)
assert rounds < cached, "the cache held %d rounds and the file as many, so the writer did not fail" % cached
print("the writer failed at the limit of %d KiB: the file holds %d rounds and the setting of round %d, the cache held %d"
      % (size_limit_kib, rounds, counter, cached))

## Files that are no store

The open refuses a file that is not SQLite, a SQLite file of another application, one with tables
but no application id, a store of a newer schema version and one of a version below 0, each with
the message of `okf/store.md`, and leaves each file as it was, byte for byte. An empty file is a
store of version 0, which the open migrates to version 1, and a file in a directory that does not
exist is refused.

In [ ]:
REFUSE = SCRATCH / "refuse"
REFUSE.mkdir()
(REFUSE / "text.db").write_text("these are the rules of the server\n", encoding="utf-8")
with sqlite3.connect(REFUSE / "other.db") as connection:
    connection.execute("PRAGMA application_id = 7")
    connection.execute("CREATE TABLE things (x)")
with sqlite3.connect(REFUSE / "tables.db") as connection:
    connection.execute("CREATE TABLE things (x)")
for name, version in (("newer.db", 2), ("negative.db", -1)):
    answers, _ = steps([open_store("refuse/" + name), CLOSE])
    connection = sqlite3.connect(REFUSE / name)
    connection.execute("PRAGMA user_version = %d" % version)
    connection.close()
(REFUSE / "empty.db").write_bytes(b"")
digest = lambda name: hashlib.sha256((REFUSE / name).read_bytes()).hexdigest()
FILES = {
    "text.db": "the file is not a SQLite database",
    "other.db": "the file is a SQLite database of another application, with the application id 0x7",
    "tables.db": "the file is a SQLite database with 1 tables and indexes but no store of Golem",
    "newer.db": "the store has the schema version 2, newer than the version 1 this Golem knows",
    "negative.db": "the store has the schema version -1, which no Golem writes",
}
before = {name: digest(name) for name in FILES}
REFUSALS = [({"op": "open", "path": "refuse/" + name}, Fails("open", "cannot open the store refuse/%s: %s" % (name, why)))
            for name, why in FILES.items()]
answers, _ = steps(REFUSALS + [
    ({"op": "open", "path": "refuse/missing/store.db"},
     Fails("open", pattern=r"cannot open the store refuse/missing/store\.db: the file cannot be opened: .+")),
    open_store("refuse/empty.db"),
    CLOSE,
])
after = {name: digest(name) for name in FILES}
assert before == after, "a refused file changed: %s" % [name for name in FILES if before[name] != after[name]]
described = json.loads(devtool("store-describe", "refuse/empty.db").stdout)
assert (described["version"], described["application_id"]) == (1, 0x476F6C6D), described

## The cost of the calls of the server thread

`golem-devtool store-bench` fills a new store with 5000 players, each with three homes, two mails,
an account and an ignore, with 50 warps, three spawns, 20 settings and eleven groups of 20 nodes,
opens it again and times 2000 calls of each kind one by one. The devtool is a debug build. Every
kind must stay below the budget of a call on the server thread in its mean and in its 99th
percentile, and the writer behind the changes runs on its own thread, so its time does not count
against the server thread.

In [ ]:
result = devtool("store-bench", "--path", "bench.db", "--players", bench_players, "--repeat", bench_repeat)
assert result.returncode == 0, result.stderr
BENCH = json.loads(result.stdout)
assert BENCH["check_passed"], BENCH
print("%-20s %9s %9s %9s %9s" % ("call", "mean us", "median us", "p99 us", "max us"))
for name, cost in BENCH["calls"].items():
    print("%-20s %9.2f %9.2f %9.2f %9.2f" % (name, cost["mean_us"], cost["median_us"], cost["p99_us"], cost["max_us"]))
over = [name for name, cost in BENCH["calls"].items() if cost["mean_us"] >= budget_us or cost["p99_us"] >= budget_us]
assert not over, "over the budget of %d us: %s" % (budget_us, over)
writes = ["join", "set_back", "set_afk", "set_home", "send_mail", "mark_mail_read", "transfer", "deposit", "ignore",
          "set_setting"]
server_ms = sum(BENCH["calls"][name]["mean_us"] * BENCH["calls"][name]["calls"] for name in writes) / 1000
print("open of %d players: %.1f ms, file %.1f MB, the process %s with the store open" % (
    bench_players, BENCH["open_ms"], BENCH["file_bytes"] / 1e6,
    "%.1f MB resident" % (BENCH["resident_kib_after_open"] / 1024) if BENCH["resident_kib_after_open"] else "of unknown size"))
print("the %d changes of the timed calls cost the server thread %.1f ms and the writer %.1f ms in %d transactions on "
      "its own thread" % (BENCH["writer"]["written"], server_ms, BENCH["writer"]["busy_ms"], BENCH["writer"]["batches"]))

## Verdict

The cell clears the temporary directory. A run that reaches it passed.

In [ ]:
import shutil

shutil.rmtree(SCRATCH)
slowest = max(BENCH["calls"].items(), key=lambda item: item[1]["mean_us"])
print("verdict: golem-store keeps the schema of %s, answered %d steps of %d scripts as expected with %d refusals, kept "
      "a prefix of its changes after a kill and after a failed writer, refused 6 files that are no store, and its "
      "slowest call of the server thread, %s, took %.1f us in the mean" % (
          wiki, COUNTS["steps"], COUNTS["scripts"], COUNTS["refusals"], slowest[0], slowest[1]["mean_us"]))